RFM Analysis


Import Libraries

In [1]:
import pandas as pd
import numpy as np 
import datetime as dt

Load Cleaned data


In [2]:
try:
    df = pd.read_excel("../data/cleaned_data.xlsx")
    print(df.head)
except Exception as e:
    print(e)

<bound method NDFrame.head of        Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
999995  576840     82582            AREA PATROLLED METAL SIGN         2   
999996  576840     82583                 HOT BATHS METAL SIGN         1   
999997  576840    82616B        FRAPPUCINO SCARF KNITTING KIT         2   
999998  576840    82616C  MIDNIGHT GLAMOUR SCARF KNITTING KIT         5   
999999  576840    84032A   CHARLIE+LOLA PINK HOT WATER BOTTLE         3   

               InvoiceDate  Price  Customer ID         Country  \
0  

In [3]:
df["TotalAmount"] = df["Quantity"] * df["Price"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days = 1)

In [5]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date-x.max()).days,
    "Invoice": "nunique",
    "TotalAmount": "sum"
}).reset_index()

rfm.columns = ['CustomerID','Recency','Frequency','Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,303,17,-64.68
1,12347.0,17,7,5408.50
2,12348.0,53,5,2019.40
3,12349.0,385,4,2646.99
4,12350.0,287,1,334.40


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm['Churn'] = (rfm['Recency'] > 90).astype(int)

print("--- Cluster-wise Customer Count ---")
print(rfm['Cluster'].value_counts())

print("\n--- Churn Count (1 = Churned, 0 = Active) ---")
print(rfm['Churn'].value_counts())

rfm.head()

--- Cluster-wise Customer Count ---
Cluster
0    3723
1    2068
2      41
3       4
Name: count, dtype: int64

--- Churn Count (1 = Churned, 0 = Active) ---
Churn
1    3130
0    2706
Name: count, dtype: int64


,CustomerID,Recency,Frequency,Monetary,Cluster,Churn
0,12346.0,303,17,-64.68,1,1
1,12347.0,17,7,5408.50,0,0
2,12348.0,53,5,2019.40,0,0
3,12349.0,385,4,2646.99,1,1
4,12350.0,287,1,334.40,1,1


Churn Model

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rfm['Avg_Order_Value'] = rfm['Monetary'] / rfm['Frequency']

X = rfm[['Frequency', 'Monetary', 'Avg_Order_Value', 'Cluster']]
y = rfm['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("--- Model Accuracy ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

rfm.to_csv('../data/customer_segments.csv', index=False)
joblib.dump(model, '../model/churn_model.pkl')
print("\n✅ Success: 'customer_segments.csv' aur 'churn_model.pkl' save ho chuke hain!")

--- Model Accuracy ---
Accuracy: 77.40%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.73      0.80      0.77       541
           1       0.82      0.75      0.78       627

    accuracy                           0.77      1168
   macro avg       0.77      0.78      0.77      1168
weighted avg       0.78      0.77      0.77      1168


✅ Success: 'customer_segments.csv' aur 'churn_model.pkl' save ho chuke hain!
